# Proximity Measures for Different Attribute Types

This notebook demonstrates how to calculate proximity measures for different types of attributes using Python packages, with minimal code.

## Overview
- **Nominal Attributes**: Categorical data without order (e.g., color, city)
- **Ordinal Attributes**: Categorical data with order (e.g., ratings, grades)
- **Numerical Attributes**: Continuous/discrete numeric data
- **Mixed Attributes**: Combination of different attribute types
- **Cosine Similarity**: Measure similarity between vectors

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from scipy.spatial.distance import pdist, squareform, cosine, hamming
from sklearn.metrics.pairwise import pairwise_distances, cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Proximity Measure for Nominal Attributes

**Concept**: For nominal attributes, similarity is measured using matching/non-matching values.
- Similarity = 1 if values match, 0 otherwise (Hamming distance)
- Dissimilarity = 1 - Similarity

In [2]:
# Example data - Nominal Attributes (Colors)
nominal_data = pd.DataFrame({
    'Object': ['A', 'B', 'C', 'D'],
    'Color': ['Red', 'Blue', 'Red', 'Green']
})

print("Nominal Data:")
print(nominal_data)
print("\n" + "="*50)

# Encode nominal attributes
le = LabelEncoder()
nominal_encoded = le.fit_transform(nominal_data['Color']).reshape(-1, 1)

# Calculate Hamming distance (dissimilarity for nominal)
hamming_dist = squareform(pdist(nominal_encoded, metric='hamming'))

# Convert to similarity (1 - distance)
nominal_similarity = 1 - hamming_dist

print("\nNominal Similarity Matrix:")
print(pd.DataFrame(nominal_similarity, 
                   index=nominal_data['Object'], 
                   columns=nominal_data['Object']))
print("\nInterpretation: 1.0 = Same color, 0.0 = Different color")

Nominal Data:
  Object  Color
0      A    Red
1      B   Blue
2      C    Red
3      D  Green


Nominal Similarity Matrix:
Object    A    B    C    D
Object                    
A       1.0  0.0  1.0  0.0
B       0.0  1.0  0.0  0.0
C       1.0  0.0  1.0  0.0
D       0.0  0.0  0.0  1.0

Interpretation: 1.0 = Same color, 0.0 = Different color


## 2. Proximity Measure for Ordinal Attributes

**Concept**: For ordinal attributes, the order matters. We rank values and calculate distance based on rank difference.
- Normalized distance based on the range of values

In [3]:
# Example data - Ordinal Attributes (Ratings)
ordinal_data = pd.DataFrame({
    'Object': ['Product_A', 'Product_B', 'Product_C', 'Product_D'],
    'Rating': [5, 3, 5, 1]  # Ordinal: 1-5 stars
})

print("Ordinal Data:")
print(ordinal_data)
print("\n" + "="*50)

# Calculate pairwise distances for ordinal data
# Normalize the ordinal values
ratings = ordinal_data['Rating'].values.reshape(-1, 1)
rating_min, rating_max = ratings.min(), ratings.max()
ratings_normalized = (ratings - rating_min) / (rating_max - rating_min)

# Calculate Euclidean distance (works well for ordinal)
ordinal_distance = pairwise_distances(ratings_normalized, metric='euclidean')

# Convert to similarity
ordinal_similarity = 1 - ordinal_distance

print("\nOrdinal Similarity Matrix (Ratings):")
print(pd.DataFrame(ordinal_similarity, 
                   index=ordinal_data['Object'], 
                   columns=ordinal_data['Object']).round(3))
print("\nInterpretation: Closer ratings = Higher similarity")

Ordinal Data:
      Object  Rating
0  Product_A       5
1  Product_B       3
2  Product_C       5
3  Product_D       1


Ordinal Similarity Matrix (Ratings):
Object     Product_A  Product_B  Product_C  Product_D
Object                                               
Product_A        1.0        0.5        1.0        0.0
Product_B        0.5        1.0        0.5        0.5
Product_C        1.0        0.5        1.0        0.0
Product_D        0.0        0.5        0.0        1.0

Interpretation: Closer ratings = Higher similarity


## 3. Proximity Measure for Numerical Attributes

**Concept**: For numerical attributes, we use distance metrics like Euclidean, Manhattan, or Minkowski.
- Most common: Euclidean distance

In [4]:
# Example data - Numerical Attributes
numerical_data = pd.DataFrame({
    'Object': ['Student_A', 'Student_B', 'Student_C', 'Student_D'],
    'Math_Score': [85.5, 90.2, 85.0, 75.3],
    'Science_Score': [88.0, 92.1, 87.5, 78.0]
})

print("Numerical Data:")
print(numerical_data)
print("\n" + "="*50)

# Extract numerical features
X = numerical_data[['Math_Score', 'Science_Score']].values

# Calculate Euclidean distance
euclidean_dist = pairwise_distances(X, metric='euclidean')

# Convert to similarity
numerical_similarity = 1 / (1 + euclidean_dist)

print("\nNumerical Similarity Matrix (Euclidean):")
print(pd.DataFrame(numerical_similarity, 
                   index=numerical_data['Object'], 
                   columns=numerical_data['Object']).round(3))

# Also show distance
print("\nEuclidean Distance Matrix:")
print(pd.DataFrame(euclidean_dist, 
                   index=numerical_data['Object'], 
                   columns=numerical_data['Object']).round(3))
print("\nInterpretation: Closer points = Higher similarity")

Numerical Data:
      Object  Math_Score  Science_Score
0  Student_A        85.5           88.0
1  Student_B        90.2           92.1
2  Student_C        85.0           87.5
3  Student_D        75.3           78.0


Numerical Similarity Matrix (Euclidean):
Object     Student_A  Student_B  Student_C  Student_D
Object                                               
Student_A      1.000      0.138      0.586      0.065
Student_B      0.138      1.000      0.126      0.046
Student_C      0.586      0.126      1.000      0.069
Student_D      0.065      0.046      0.069      1.000

Euclidean Distance Matrix:
Object     Student_A  Student_B  Student_C  Student_D
Object                                               
Student_A      0.000      6.237      0.707     14.284
Student_B      6.237      0.000      6.943     20.514
Student_C      0.707      6.943      0.000     13.577
Student_D     14.284     20.514     13.577      0.000

Interpretation: Closer points = Higher similarity


## 4. Proximity Measure for Mixed Attributes

**Concept**: When data has both nominal, ordinal, and numerical attributes, we use combined metrics.
- Normalize different attribute types and combine them

In [5]:
# Example data - Mixed Attributes
mixed_data = pd.DataFrame({
    'ID': ['Person_1', 'Person_2', 'Person_3', 'Person_4'],
    'City': ['New York', 'Los Angeles', 'New York', 'Chicago'],  # Nominal
    'Education': ['High School', 'Bachelor', 'Bachelor', 'Master'],  # Ordinal
    'Age': [25, 35, 28, 45]  # Numerical
})

print("Mixed Attributes Data:")
print(mixed_data)
print("\n" + "="*50)

# Encode categorical variables
le_city = LabelEncoder()
le_education = LabelEncoder()

mixed_encoded = pd.DataFrame({
    'City_encoded': le_city.fit_transform(mixed_data['City']),
    'Education_encoded': le_education.fit_transform(mixed_data['Education']),
    'Age': mixed_data['Age']
})

# Normalize all features
mixed_normalized = (mixed_encoded - mixed_encoded.min()) / (mixed_encoded.max() - mixed_encoded.min())

# Calculate mixed distance (using Manhattan for mixed types)
mixed_distance = pairwise_distances(mixed_normalized.values, metric='manhattan')

# Normalize distance
mixed_distance_normalized = mixed_distance / mixed_distance.max()

# Convert to similarity
mixed_similarity = 1 - mixed_distance_normalized

print("\nMixed Attributes Similarity Matrix:")
print(pd.DataFrame(mixed_similarity, 
                   index=mixed_data['ID'], 
                   columns=mixed_data['ID']).round(3))
print("\nInterpretation: Combines nominal, ordinal, and numerical similarities")

Mixed Attributes Data:
         ID         City    Education  Age
0  Person_1     New York  High School   25
1  Person_2  Los Angeles     Bachelor   35
2  Person_3     New York     Bachelor   28
3  Person_4      Chicago       Master   45


Mixed Attributes Similarity Matrix:
ID        Person_1  Person_2  Person_3  Person_4
ID                                              
Person_1     1.000     0.474     0.772     0.123
Person_2     0.474     1.000     0.702     0.298
Person_3     0.772     0.702     1.000     0.000
Person_4     0.123     0.298     0.000     1.000

Interpretation: Combines nominal, ordinal, and numerical similarities


## 5. Cosine Similarity

**Concept**: Cosine similarity measures the cosine of the angle between two vectors.
- Range: -1 to 1 (or 0 to 1 for non-negative vectors)
- 1 = same direction, 0 = orthogonal, -1 = opposite direction

In [6]:
# Example data - Vector representation (e.g., text documents, user preferences)
cosine_data = pd.DataFrame({
    'Document': ['Doc_A', 'Doc_B', 'Doc_C', 'Doc_D'],
    'Word_1': [5, 2, 3, 1],
    'Word_2': [4, 1, 5, 2],
    'Word_3': [3, 5, 2, 4]
})

print("Document Data (Term Frequency):")
print(cosine_data)
print("\n" + "="*50)

# Extract feature vectors
X_cosine = cosine_data[['Word_1', 'Word_2', 'Word_3']].values

# Calculate cosine similarity using sklearn
cosine_sim_matrix = cosine_similarity(X_cosine)

print("\nCosine Similarity Matrix:")
print(pd.DataFrame(cosine_sim_matrix, 
                   index=cosine_data['Document'], 
                   columns=cosine_data['Document']).round(3))

# Also show cosine distance
cosine_dist_matrix = pairwise_distances(X_cosine, metric='cosine')

print("\nCosine Distance Matrix (1 - Similarity):")
print(pd.DataFrame(cosine_dist_matrix, 
                   index=cosine_data['Document'], 
                   columns=cosine_data['Document']).round(3))
print("\nInterpretation: 1.0 = Identical vectors, 0.0 = Orthogonal vectors")

Document Data (Term Frequency):
  Document  Word_1  Word_2  Word_3
0    Doc_A       5       4       3
1    Doc_B       2       1       5
2    Doc_C       3       5       2
3    Doc_D       1       2       4


Cosine Similarity Matrix:
Document  Doc_A  Doc_B  Doc_C  Doc_D
Document                            
Doc_A     1.000  0.749  0.941  0.772
Doc_B     0.749  1.000  0.622  0.956
Doc_C     0.941  0.622  1.000  0.743
Doc_D     0.772  0.956  0.743  1.000

Cosine Distance Matrix (1 - Similarity):
Document  Doc_A  Doc_B  Doc_C  Doc_D
Document                            
Doc_A     0.000  0.251  0.059  0.228
Doc_B     0.251  0.000  0.378  0.044
Doc_C     0.059  0.378  0.000  0.257
Doc_D     0.228  0.044  0.257  0.000

Interpretation: 1.0 = Identical vectors, 0.0 = Orthogonal vectors


## Summary Table

| Attribute Type | Distance Metric | Use Case | Python Package |
|---|---|---|---|
| **Nominal** | Hamming | Categorical data | `scipy.spatial.distance` |
| **Ordinal** | Euclidean (normalized) | Ranked data | `sklearn.metrics.pairwise_distances` |
| **Numerical** | Euclidean/Manhattan | Continuous data | `sklearn.metrics.pairwise_distances` |
| **Mixed** | Manhattan (normalized) | Combined attributes | Custom normalization |
| **Cosine** | Cosine | Vector similarity | `sklearn.metrics.pairwise.cosine_similarity` |

### Key Takeaways:
✅ Use pre-built packages for efficiency and accuracy  
✅ Normalize data before computing distances for mixed attributes  
✅ Choose metric based on attribute type  
✅ Convert distance to similarity using: Similarity = 1 - Normalized_Distance

In [7]:
# Quick Reference - One-line functions for each proximity measure

print("QUICK REFERENCE CODE:")
print("="*60)
print("\n1. NOMINAL ATTRIBUTES:")
print("   similarity = 1 - squareform(pdist(encoded_data, metric='hamming'))")

print("\n2. ORDINAL ATTRIBUTES:")
print("   similarity = 1 - pairwise_distances(normalized_data, metric='euclidean')")

print("\n3. NUMERICAL ATTRIBUTES:")
print("   similarity = 1 / (1 + pairwise_distances(data, metric='euclidean'))")

print("\n4. MIXED ATTRIBUTES:")
print("   similarity = 1 - (pairwise_distances(norm_data, metric='manhattan') / max)")

print("\n5. COSINE SIMILARITY:")
print("   similarity = cosine_similarity(data)")
print("\n" + "="*60)

QUICK REFERENCE CODE:

1. NOMINAL ATTRIBUTES:
   similarity = 1 - squareform(pdist(encoded_data, metric='hamming'))

2. ORDINAL ATTRIBUTES:
   similarity = 1 - pairwise_distances(normalized_data, metric='euclidean')

3. NUMERICAL ATTRIBUTES:
   similarity = 1 / (1 + pairwise_distances(data, metric='euclidean'))

4. MIXED ATTRIBUTES:
   similarity = 1 - (pairwise_distances(norm_data, metric='manhattan') / max)

5. COSINE SIMILARITY:
   similarity = cosine_similarity(data)

